In [23]:
import torch
from torch import nn, optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [24]:
tf = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
    transforms.ColorJitter(brightness=(0.8, 1.2))
])

dataset = datasets.ImageFolder(root = "data/Training", transform=tf)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_dl = DataLoader(
    train_dataset,
    batch_size = 32,
    shuffle= True,
    num_workers= 4,
    pin_memory=True
)

val_dl = DataLoader(
    train_dataset,
    batch_size = 32,
    shuffle= False,
    num_workers= 4,
    pin_memory=True
)

test_dl = DataLoader(
    datasets.ImageFolder("data/Testing", transform=tf),
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [25]:
import torchvision.models as models

class TumorModel(nn.Module):
    def __init__(self, freeze_backbone = True):
        super().__init__()
        self.base_model = models.efficientnet_b1(weights="IMAGENET1K_V1")
        in_features = self.base_model.classifier[1].in_features
        self.base_model.classifier = nn.Identity()

        if freeze_backbone:
            for param in self.base_model.parameters():
                param.requires_grad = False

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(in_features = in_features, out_features=128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(in_features = 128, out_features = 4)
        )
    def forward(self, x):
        x = self.base_model(x)
        x = self.classifier(x)
        return x

In [26]:
model_2 = TumorModel().to(device)

In [27]:
optimizer = optim.Adam(params=model_2.parameters(), lr= 0.001)
loss_fn = nn.CrossEntropyLoss()

In [28]:
for epoch in range(11):
    model_2.train()
    train_loss, train_correct = 0.0, 0
    for images, labels in train_dl:
        images, labels = images.to(device), labels.to(device)
        logit = model_2(images)

        optimizer.zero_grad()
        loss = loss_fn(logit, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        _, pred = torch.max(logit, dim = 1)
        train_correct += (pred == labels).sum().item()
    train_acc = 100 * train_correct/len(train_dl.dataset)
    train_loss /= len(train_dl.dataset)

    model_2.eval()
    val_loss, val_correct = 0.0, 0
    with torch.no_grad():
        for images, labels in val_dl:
            images, labels = images.to(device), labels.to(device)
            logit = model_2(images)

            loss = loss_fn(logit, labels)
            val_loss += loss.item() * images.size(0)
            _, pred = torch.max(logit, dim = 1)
            val_correct += (pred == labels).sum().item()
    val_accuracy = 100 * val_correct/len(val_dl.dataset)
    val_loss /= len(val_dl.dataset)

    print(f"Epoch: {epoch} | TrainLoss: {train_loss} | TrainAcc: {train_acc} | ValLoss: {val_loss} | ValAcc: {val_accuracy}")

Epoch: 0 | TrainLoss: 0.8233130805261236 | TrainAcc: 67.8485445392865 | ValLoss: 0.6311018885186476 | ValAcc: 79.14204421098708
Epoch: 1 | TrainLoss: 0.6566620211315302 | TrainAcc: 74.5458524841322 | ValLoss: 0.8050574407480456 | ValAcc: 81.4182534471438
Epoch: 2 | TrainLoss: 0.6096843051477292 | TrainAcc: 76.07791639308383 | ValLoss: 0.5456385284133907 | ValAcc: 82.88465747428322
Epoch: 3 | TrainLoss: 0.5919665395104668 | TrainAcc: 76.66885532939374 | ValLoss: 0.6388895961629872 | ValAcc: 84.63558765594222
Epoch: 4 | TrainLoss: 0.5469655137819752 | TrainAcc: 78.17903261107463 | ValLoss: 0.4092859711459701 | ValAcc: 86.32085795578901
Epoch: 5 | TrainLoss: 0.5285414540942479 | TrainAcc: 80.19260231998248 | ValLoss: 0.37274300770637997 | ValAcc: 87.04311665572335
Epoch: 6 | TrainLoss: 0.518634843491026 | TrainAcc: 79.36091048369447 | ValLoss: 0.35598144254490854 | ValAcc: 88.55329393740425
Epoch: 7 | TrainLoss: 0.49951686647990307 | TrainAcc: 80.8929743926461 | ValLoss: 0.574611885930130

In [31]:
model_2.eval()
test_loss, test_correct = 0.0, 0
with torch.no_grad():
    for images, labels in test_dl:
        images, labels = images.to(device), labels.to(device)
        
        logits = model_2(images)
        _, output = torch.max(logits, dim = 1)
        loss = loss_fn(logits, labels)

        test_loss += loss.item() * images.size(0)
        test_correct += (output == labels).sum().item()
test_accuracy = 100 * test_correct/len(test_dl.dataset)
test_loss /= len(test_dl.dataset)
print(f"Test Accuracy: {test_accuracy} | Test Loss: {test_loss}")


Test Accuracy: 82.53241800152556 | Test Loss: 0.46467116573423395


In [32]:
torch.save(model_2.state_dict(), "models/model_2.pth")